# Ekstraksi Fitur Citra Sandal (Metode Klasik)
## Klasifikasi Kondisi Sandal: Rapi vs Berantakan

Notebook ini berfokus pada **ekstraksi fitur citra digital secara klasik** (*handcrafted features*) yang mencakup:
1. **Fitur Warna (6 fitur)**: Rata-rata (*Mean*) & Standar Deviasi (*Standard Deviation*) kanal Red, Green, Blue.
2. **Fitur Tepi & Gradien (2 fitur)**: Laplacian Variance (ketajaman tepi) & Canny Edge Density (kepadatan tepi kontur).
3. **Fitur Tekstur Spasial GLCM (4 fitur)**: Kontras, Homogenitas, Energi, dan Korelasi spasial matriks GLCM.

Hasil ekstraksi fitur akan disimpan ke dalam berkas `hasil_ekstraksi/fitur_rgb_std.csv`.

## 1. Import Library yang Dibutuhkan
Memanggil pustaka utama untuk pemrosesan citra digital dan analisis tabular:
- **`cv2` (OpenCV)**: Membaca gambar, manipulasi channel warna, konversi grayscale, dan deteksi tepi (Canny & Laplacian).
- **`skimage.feature`**: Ekstraksi fitur tekstur spasial GLCM (*Gray-Level Co-occurrence Matrix*).
- **`numpy`**: Perhitungan statistik matriks dan array piksel (mean, standar deviasi).
- **`pandas`**: Mengelola dataset hasil ekstraksi ke dalam bentuk tabular (DataFrame) dan ekspor ke file CSV.
- **`matplotlib.pyplot`**: Visualisasi komparasi citra gambar.

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops
import matplotlib.pyplot as plt

# Deteksi direktori proyek secara otomatis
current_path = os.getcwd()
if os.path.exists(os.path.join(current_path, 'data')):
    BASE_DIR = os.path.join(current_path, 'data')
    OUTPUT_DIR = os.path.join(current_path, 'hasil_ekstraksi')
elif os.path.exists(os.path.join(os.path.dirname(current_path), 'data')):
    BASE_DIR = os.path.join(os.path.dirname(current_path), 'data')
    OUTPUT_DIR = os.path.join(os.path.dirname(current_path), 'hasil_ekstraksi')
else:
    BASE_DIR = r'D:\KULIAH\SEMESTER 5\psd\uas\ProjectImageClasification\data'
    OUTPUT_DIR = r'D:\KULIAH\SEMESTER 5\psd\uas\ProjectImageClasification\hasil_ekstraksi'

CLASSES = ["Berantakan", "Rapi"]
print(f"Direktori Dataset: {BASE_DIR}")
print(f"Kelas Target: {CLASSES}")

## 2. Ekstraksi Fitur Citra (Warna, Tepi, dan Tekstur GLCM)
Melakukan perulangan (*looping*) ke seluruh gambar pada folder kelas untuk mengekstrak **12 fitur digital**.

In [2]:
data_fitur = []

for label in CLASSES:
    folder_path = os.path.join(BASE_DIR, label)
    if not os.path.exists(folder_path):
        continue
        
    file_list = os.listdir(folder_path)
    for filename in file_list:
        if not filename.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
            continue
            
        img_path = os.path.join(folder_path, filename)
        img = cv2.imread(img_path)
        if img is None:
            continue
            
        # 1. Fitur Warna RGB (Mean & Standar Deviasi)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        r, g, b = img_rgb[:,:,0], img_rgb[:,:,1], img_rgb[:,:,2]
        mean_r, mean_g, mean_b = np.mean(r), np.mean(g), np.mean(b)
        std_r, std_g, std_b = np.std(r), np.std(g), np.std(b)
        
        # 2. Fitur Tepi & Gradien (Laplacian Variance & Canny Density)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
        edges_canny = cv2.Canny(gray, 100, 200)
        edge_density = np.sum(edges_canny > 0) / edges_canny.size
        
        # 3. Fitur Tekstur Spasial GLCM
        glcm = graycomatrix(gray, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
        contrast = graycoprops(glcm, 'contrast')[0, 0]
        homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
        energy = graycoprops(glcm, 'energy')[0, 0]
        correlation = graycoprops(glcm, 'correlation')[0, 0]
        
        data_fitur.append({
            'filename': filename,
            'label': label.lower(),
            'mean_red': round(mean_r, 4),
            'mean_green': round(mean_g, 4),
            'mean_blue': round(mean_b, 4),
            'std_red': round(std_r, 4),
            'std_green': round(std_g, 4),
            'std_blue': round(std_b, 4),
            'edge_laplacian_variance': round(laplacian_var, 4),
            'edge_density': round(edge_density, 4),
            'glcm_contrast': round(contrast, 4),
            'glcm_homogeneity': round(homogeneity, 4),
            'glcm_energy': round(energy, 4),
            'glcm_correlation': round(correlation, 4)
        })

print(f"Selesai mengekstrak {len(data_fitur)} citra!")

## 3. Membuat DataFrame dan Menyimpan ke Berkas CSV
Menyusun data hasil ekstraksi ke dalam bentuk DataFrame pandas lalu mengekspor ke `hasil_ekstraksi/fitur_rgb_std.csv`.

In [3]:
df = pd.DataFrame(data_fitur)
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_csv = os.path.join(OUTPUT_DIR, 'fitur_rgb_std.csv')
df.to_csv(output_csv, index=False)
print(f"Data berhasil disimpan ke: {output_csv}")
df.head()

## 4. Visualisasi Citra: Sampel Rapi vs Berantakan (RGB & Grayscale)
Menampilkan visualisasi komparasi gambar sandal dari kedua kelas dalam format Grid 2x2 secara dinamis:
- **Baris 1**: Sampel citra kelas **Rapi** dalam format warna (RGB) dan derajat keabuan (Grayscale).
- **Baris 2**: Sampel citra kelas **Berantakan** dalam format RGB dan Grayscale.

In [4]:
sample_rapi_path = os.path.join(BASE_DIR, "Rapi", "rapi_001.jpg")
sample_berantakan_path = os.path.join(BASE_DIR, "Berantakan", "berantakan_001.jpg")

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

img_rapi_bgr = cv2.imread(sample_rapi_path)
img_rapi_rgb = cv2.cvtColor(img_rapi_bgr, cv2.COLOR_BGR2RGB)
img_rapi_gray = cv2.cvtColor(img_rapi_bgr, cv2.COLOR_BGR2GRAY)

img_ber_bgr = cv2.imread(sample_berantakan_path)
img_ber_rgb = cv2.cvtColor(img_ber_bgr, cv2.COLOR_BGR2RGB)
img_ber_gray = cv2.cvtColor(img_ber_bgr, cv2.COLOR_BGR2GRAY)

# Row 1: Rapi
axes[0, 0].imshow(img_rapi_rgb)
axes[0, 0].set_title("Sampel Rapi (RGB)", fontsize=12, fontweight='bold', color='green')
axes[0, 0].axis('off')

axes[0, 1].imshow(img_rapi_gray, cmap='gray')
axes[0, 1].set_title("Sampel Rapi (Grayscale)", fontsize=12, fontweight='bold')
axes[0, 1].axis('off')

# Row 2: Berantakan
axes[1, 0].imshow(img_ber_rgb)
axes[1, 0].set_title("Sampel Berantakan (RGB)", fontsize=12, fontweight='bold', color='red')
axes[1, 0].axis('off')

axes[1, 1].imshow(img_ber_gray, cmap='gray')
axes[1, 1].set_title("Sampel Berantakan (Grayscale)", fontsize=12, fontweight='bold')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()